In [21]:
# This script takes the predicted loads as input (Step 1 from 2-step approach)
# The power predictions from the models (tgt, Sarima, Xgboost, naive) are used as active Power
# The q_mvar var is added by using the case and bus specific scaling factor.
# Output: One csv for each model prediction
# Next step: input the csv files into datakit to retrieve the OPF solution for Step 2 of 2-step approach 

In [22]:
import os
import numpy as np
import pandas as pd

CASE = 118                   # set IEEE case
PARQUET = "../data/data_in/case118_ieee_horizon24.parquet"
OUT_ROOT = "../data/precomputed_profiles/"
ID_COL = "load_scenario_idx"

In [23]:
# cell 2 - helper: load IEEE base P/Q from the PGLib .m cases shipped with gridfm-datakit
import sys
from pathlib import Path
from importlib import resources

from matpowercaseframes import CaseFrames


def _find_gridfm_datakit_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "gridfm_datakit").is_dir():
            return p
    return start


GRIDFM_DATAKIT_ROOT = _find_gridfm_datakit_root(Path.cwd().resolve())
if str(GRIDFM_DATAKIT_ROOT) not in sys.path:
    sys.path.insert(0, str(GRIDFM_DATAKIT_ROOT))


def get_ieee_base(case_n: int):
    """Return (P_base, Q_base) arrays for `pglib_opf_case{case_n}_ieee.m`."""
    filename = f"pglib_opf_case{case_n}_ieee.m"
    m_path = Path(str(resources.files("gridfm_datakit.grids").joinpath(filename)))
    if not m_path.is_file():
        raise FileNotFoundError(f"Missing Matpower case file: {m_path}")
    cf = CaseFrames(str(m_path))
    P_base = cf.bus["PD"].to_numpy(dtype=float)
    Q_base = cf.bus["QD"].to_numpy(dtype=float)
    return P_base, Q_base

In [24]:
# cell 3 - load parquet, validate, derive ratio, and build flattened scenario index
if not os.path.exists(PARQUET):
    raise FileNotFoundError(PARQUET)

df = pd.read_parquet(PARQUET)

id_col = ID_COL
if id_col not in df.columns:
    raise ValueError(f"Missing required id column: {id_col}")

# ensure horizon dimension is explicit for stable flattening
if "horizon_step" in df.columns:
    df["_horizon_step"] = pd.to_numeric(df["horizon_step"], errors="raise").astype(int)
else:
    df["_horizon_step"] = 0

# get base P/Q and compute Q/P ratio
P_base, Q_base = get_ieee_base(CASE)
P_base = np.asarray(P_base, dtype=float)
Q_base = np.asarray(Q_base, dtype=float)

ratio = np.zeros_like(P_base, dtype=float)
mask = P_base != 0.0
ratio[mask] = Q_base[mask] / P_base[mask]

# datakit expects load_scenario to be contiguous 0..N-1;
# flatten by (original scenario idx, horizon step) without changing row count
df[id_col] = pd.to_numeric(df[id_col], errors="raise").astype(int)
rows_before = len(df)
df["_load_scenario"] = df.groupby([id_col, "_horizon_step"], sort=True).ngroup()
if len(df) != rows_before:
    raise RuntimeError("Unexpected row-count change while building flattened scenarios")

# ensure `load` is the 0-based bus index used by the Matpower case file
bus_raw = pd.to_numeric(df["bus_id"], errors="raise").astype(int)
if bus_raw.min() == 1 and bus_raw.max() == len(P_base):
    df["_load_idx"] = bus_raw - 1
else:
    df["_load_idx"] = bus_raw

In [25]:
df.shape

(11098608, 10)

In [26]:
# cell 4 - transform, validate, and write one CSV per model column
exclude = {id_col, "bus_id", "horizon_step", "_horizon_step", "_load_scenario", "_load_idx"}
model_cols = [c for c in df.columns if c not in exclude]

out_dir = os.path.join(OUT_ROOT, f"case{CASE}_ieee")
os.makedirs(out_dir, exist_ok=True)

load_idx = pd.to_numeric(df["_load_idx"], errors="raise").to_numpy(dtype=int)
valid_mask = (load_idx >= 0) & (load_idx < len(ratio))
if not valid_mask.all():
    bad = int((~valid_mask).sum())
    raise ValueError(f"Found {bad} invalid load indices outside 0..{len(ratio)-1}")

n_buses = len(ratio)
n_flat_scenarios = int(df["_load_scenario"].nunique())
expected_rows = n_buses * n_flat_scenarios

if int(df["_load_scenario"].min()) != 0 or int(df["_load_scenario"].max()) != n_flat_scenarios - 1:
    raise ValueError("Flattened load_scenario is not contiguous 0..S-1")

print(
    "Diagnostics:",
    {
        "rows": int(len(df)),
        "orig_scenarios": int(df[id_col].nunique()),
        "horizons": int(df["_horizon_step"].nunique()),
        "flattened_scenarios": n_flat_scenarios,
        "buses": n_buses,
    },
)
print(
    f"Expected rows check: {n_buses} x {n_flat_scenarios} = {expected_rows}; actual rows={len(df)}"
)

for col in model_cols:
    p_arr = pd.to_numeric(df[col], errors="coerce").fillna(0.0).to_numpy(dtype=float)
    q_arr = p_arr * ratio[load_idx]
    out = pd.DataFrame({
        "load_scenario": df["_load_scenario"].astype(int),
        "load_scenario_idx": df[id_col].astype(int),
        "horizon_step": df["_horizon_step"].astype(int),
        "load": df["_load_idx"].astype(int),
        "p_mw": p_arr,
        "q_mvar": q_arr,
    })

    dup_count = int(out.duplicated(subset=["load_scenario", "load"]).sum())
    if dup_count > 0:
        raise ValueError(
            f"{col}: detected {dup_count} duplicate (load_scenario, load) pairs after flattening"
        )

    if len(out) != expected_rows:
        raise ValueError(
            f"{col}: expected {expected_rows} rows ({n_buses} buses x {n_flat_scenarios} scenarios), got {len(out)}"
        )

    out_path = os.path.join(out_dir, f"{col}.csv")
    out.to_csv(out_path, index=False)
    print(
        "wrote",
        out_path,
        "| scenarios=",
        n_flat_scenarios,
        "| rows=",
        len(out),
    )

Diagnostics: {'rows': 11098608, 'orig_scenarios': 3919, 'horizons': 24, 'flattened_scenarios': 94056, 'buses': 118}
Expected rows check: 118 x 94056 = 11098608; actual rows=11098608
wrote ../data/precomputed_profiles/case118_ieee\true.csv | scenarios= 94056 | rows= 11098608
wrote ../data/precomputed_profiles/case118_ieee\xgb.csv | scenarios= 94056 | rows= 11098608
wrote ../data/precomputed_profiles/case118_ieee\snaive.csv | scenarios= 94056 | rows= 11098608
wrote ../data/precomputed_profiles/case118_ieee\sarima.csv | scenarios= 94056 | rows= 11098608


In [27]:
df["load_scenario_idx"].nunique()

3919

In [28]:
df2=pd.read_parquet("../data/data_out/sarima/case14_ieee/raw/bus_data.parquet")

FileNotFoundError: [Errno 2] No such file or directory: '../data/data_out/sarima/case14_ieee/raw/bus_data.parquet'

In [ ]:
df2.head()